<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z324_AutoGluon_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon con Features Enriquecidas

## ¿Qué agregamos respecto a z316?

### 1. Quantiles históricos como static features
Para cada producto calculamos los quantiles de su distribución histórica de ventas:
`q10, q25, q50, q75, q90, media, desvio, coef_variacion`

Estos van como **static features** — características fijas por producto que no cambian en el tiempo. Los modelos globales de AutoGluon (DeepAR, TFT) las usan para diferenciar entre productos: uno con q90 alto es estructuralmente distinto a uno con q90 bajo.

### 2. Mes del año como known covariate
El mes (1-12) se conoce de antemano para cualquier horizonte futuro — AutoGluon lo llama `known_covariates`. Los modelos que los soportan (TFT, RecursiveTabular, DirectTabular) lo usan como regresor, capturando estacionalidad mensual.

### 3. Mejora de parámetros
- `time_limit` ajustable
- `num_val_windows` aumentado para validación más robusta
- `presets` configurable

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia.
<br>**Importante**: cambiar el numero de experimento en cada corrida — AutoGluon reutiliza modelos de la misma carpeta.

In [ ]:
PARAM = {
  'experimento': 'AutoGluon_Features-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'eval_metric': 'RMSE',
  'time_limit': 3600,       # segundos — aumentar si tenés GPU
  'num_val_windows': 3,     # z316 usaba 2, mas ventanas = validacion mas robusta
  'presets': 'best_quality'
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

print(f"{tb_ventas.height} filas, {tb_ventas['product_id'].n_unique()} productos")

# 3  Static features — quantiles históricos por producto

Para cada producto calculamos estadísticas de su distribución histórica completa.
AutoGluon las pasa a los modelos globales como contexto fijo del producto.

| Feature | Qué captura |
|---|---|
| `q10`, `q25`, `q50`, `q75`, `q90` | Forma de la distribución de ventas |
| `media` | Nivel promedio |
| `desvio` | Volatilidad absoluta |
| `cv` | Coeficiente de variación — volatilidad relativa al nivel |
| `n_meses` | Cuántos meses de historia tiene el producto |

In [ ]:
static_rows = []

for pid in tb_ventas['product_id'].unique().to_list():
    vals = (
        tb_ventas.filter(pl.col('product_id') == pid)
        .sort('periodo')['tn'].to_numpy().astype(float)
    )
    media  = vals.mean()
    desvio = vals.std()
    cv     = desvio / (media + 1e-9)

    static_rows.append({
        'item_id': pid,
        'q10':    float(np.quantile(vals, 0.10)),
        'q25':    float(np.quantile(vals, 0.25)),
        'q50':    float(np.quantile(vals, 0.50)),
        'q75':    float(np.quantile(vals, 0.75)),
        'q90':    float(np.quantile(vals, 0.90)),
        'media':  float(media),
        'desvio': float(desvio),
        'cv':     float(cv),
        'n_meses': len(vals)
    })

df_static = pd.DataFrame(static_rows).set_index('item_id')
print(f"Static features: {df_static.shape}")
display(df_static.head(5))

# 4  Known covariates — mes del año

El mes (1-12) es una variable que se conoce de antemano para cualquier fecha futura.
AutoGluon la pasa como `known_covariates` a los modelos que la soportan (TFT, RecursiveTabular, DirectTabular).

Necesita estar presente tanto en los datos de entrenamiento como en los periodos futuros a predecir.

In [ ]:
# agrego el mes como columna en tb_ventas
tb_ventas = tb_ventas.with_columns(
    pl.col('timestamp').dt.month().alias('mes')
)

display(tb_ventas.head(3))

# 5  Construccion del TimeSeriesDataFrame

AutoGluon recibe:
- `ts_data`: el TimeSeriesDataFrame con `tn` como target y `mes` como covariate
- `static_features`: el DataFrame de quantiles, indexado por `item_id`

In [ ]:
ts_data = TimeSeriesDataFrame.from_data_frame(
    tb_ventas.to_pandas(),
    timestamp_column='timestamp',
    id_column='product_id'
)

# adjuntamos las static features
ts_data.static_features = df_static

print(ts_data)
print(f"\nStatic features adjuntas: {ts_data.static_features.shape}")

# 6  Entrenamiento AutoGluon

Le indicamos que `mes` es una `known_covariates_names` — AutoGluon solo se la pasa a los modelos que la soportan, el resto la ignora graciosamente.

In [ ]:
modelo = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS',
    eval_metric=PARAM['eval_metric'],
    known_covariates_names=['mes']  # mes es conocido para el futuro
)

modelo.fit(
    ts_data,
    num_val_windows=PARAM['num_val_windows'],
    time_limit=PARAM['time_limit'],
    presets=PARAM['presets'],
    random_seed=PARAM['semilla_primigenia']
)

# 7  Prediccion

Para predecir necesitamos pasarle los valores futuros de `mes` (que son conocidos: enero=1, febrero=2).

In [ ]:
# construyo el dataframe de covariates futuras: 202001 y 202002 para cada producto
productos = tb_apredecir['product_id'].to_list()
future_rows = []
for pid in productos:
    future_rows.append({'product_id': pid, 'timestamp': datetime(2020, 1, 1), 'mes': 1})
    future_rows.append({'product_id': pid, 'timestamp': datetime(2020, 2, 1), 'mes': 2})

df_future = pd.DataFrame(future_rows)
df_future = df_future.set_index(['product_id', 'timestamp'])

tb_forecast = modelo.predict(
    ts_data,
    known_covariates=df_future,
    random_seed=PARAM['semilla_primigenia']
)

display(tb_forecast)

# 8  Armado del submit

In [ ]:
tb_forecast_pl = pl.from_pandas(tb_forecast.reset_index())

tb_final = (
    tb_forecast_pl
    .filter(pl.col('timestamp') == datetime(2020, 2, 1))
    .select(['item_id', 'mean'])
    .rename({'item_id': 'product_id', 'mean': 'tn'})
)

# negativos a cero
tb_final = tb_final.with_columns(
    pl.when(pl.col('tn') < 0).then(0.0).otherwise(pl.col('tn')).alias('tn')
)

display(tb_final)
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

# 9  Submit a Kaggle

In [ ]:
archivo = f"AutoGluon_Features_{PARAM['eval_metric']}.csv"
mensaje = f"AutoGluon quantiles_static + mes_covariate {PARAM['eval_metric']}"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 10  Que probar si el score no mejora

| Cambio | Donde | Por que |
|---|---|---|
| `num_val_windows: 4` | PARAM | Validacion mas robusta |
| `time_limit: 7200` | PARAM | Mas tiempo para modelos profundos |
| Agregar `trimestre` (1-4) | Sección 4 | Estacionalidad mas gruesa |
| Agregar `tendencia_12m` como static | Sección 3 | Pendiente de la serie como feature fija |
| Agregar `pct_ceros` como static | Sección 3 | % de meses con tn=0 — identifica productos esporadicos |